# Forex Data Analysis

This notebook reads EUR/USD exchange rate data and displays it in a table format.

In [65]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# Read the EUR/USD data from CSV file
df = pd.read_csv('./eurusd.csv')

def percentage(value, total):
    return f"{value / total * 100:.1f}%"

display(
    HTML("<h2>Profitable Trades</h2><p>Breakdown of profitable trades (1:1 RRR) by different criteria.</p>")
)

pd.DataFrame({
    'Data': [
        'Total',
        'Buy Only',
        'Sell Only',
        'With EMA',
        'Against EMA',
        'BOS Trades',
        'CH Trades',
        'With EMA & BOS',
        'With EMA & CH',
    ],
    'Value': [
        percentage(df[df['TP'] >= df['SL']].shape[0], df.shape[0]),
        percentage(df[(df['TP'] >= df['SL']) & (df['Direction'] == 'Buy')].shape[0], df.shape[0]),
        percentage(df[(df['TP'] >= df['SL']) & (df['Direction'] == 'Sell')].shape[0], df.shape[0]),
        percentage(df[(df['TP'] >= df['SL']) & (df['EMA'] == df['Direction'])].shape[0], df.shape[0]),
        percentage(df[(df['TP'] >= df['SL']) & (df['EMA'] != df['Direction'])].shape[0], df.shape[0]),
        percentage(df[(df['TP'] >= df['SL']) & (df['BOS/CH'] == 'BOS')].shape[0], df.shape[0]),
        percentage(df[(df['TP'] >= df['SL']) & (df['BOS/CH'] == 'CH')].shape[0], df.shape[0]),
        percentage(df[(df['TP'] >= df['SL']) & (df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS')].shape[0], df.shape[0]),
        percentage(df[(df['TP'] >= df['SL']) & (df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'CH')].shape[0], df.shape[0]),
    ]
})

,Data,Value
0,Total,50.0%
1,Buy Only,25.0%
2,Sell Only,25.0%
3,With EMA,33.3%
4,Against EMA,16.7%
5,BOS Trades,37.5%
6,CH Trades,12.5%
7,With EMA & BOS,29.2%
8,With EMA & CH,4.2%


In [66]:
# Display the data table for debugging reasons
df

,Date,Trade,Weekday,Hour,Direction,EMA,SL,Pullback,TP,BOS/CH,30M\nLeg,HH Until\nNews,News\nEvent
0,2025-08-29,#1,Friday,10,Buy,Sell,4.0,2.0,25,CH,Below H,5.0,Core PCE
1,2025-08-29,#2,Friday,12,Buy,Buy,5.9,1.1,6,BOS,Below H,3.0,Core PCE
2,2025-08-29,#3,Friday,14,Sell,Sell,3.9,2.1,13,BOS,Below H,1.0,Core PCE
3,2025-08-29,#4,Friday,14,Sell,Sell,2.3,2.3,0,BOS,Below H,1.0,Core PCE
4,2025-08-29,#5,Friday,16,Buy,Sell,7.9,4.3,48,BOS,Above L,NaN,NaN
5,2025-09-01,#1,Monday,10,Buy,Buy,3.4,3.4,0,CH,Above H,NaN,NaN
6,2025-09-01,#2,Monday,10,Sell,Buy,5.7,5.7,0,BOS,Above H,NaN,NaN
7,2025-09-01,#3,Monday,11,Buy,Buy,5.1,4.4,15,CH,Above H,NaN,NaN
8,2025-09-01,#4,Monday,13,Buy,Sell,2.6,0.5,10,BOS,Below H,NaN,NaN
9,2025-09-01,#5,Monday,15,Sell,Sell,1.4,1.2,8,BOS,Below L,NaN,NaN


In [67]:
# Define RRR calculation function
def calculate_rrr_stats(data_df, strategy_name):
    """
    Calculate Risk-Reward Ratio statistics for a given dataframe and strategy.
    
    Args:
        data_df: DataFrame containing trading data
        strategy_name: Name of the strategy for labeling
    
    Returns:
        DataFrame with RRR statistics
    """
    total_trades = len(data_df)
    
    rrr_configs = [
        (1, 50.0),   # 1:1 RRR needs 50% win rate to break even
        (2, 33.0),   # 1:2 RRR needs 33% win rate to break even
        (3, 25.0),   # 1:3 RRR needs 25% win rate to break even
        (4, 20.0),   # 1:4 RRR needs 20% win rate to break even
        (5, 17.0),   # 1:5 RRR needs 17% win rate to break even
        (10, 9.0),   # 1:10 RRR needs 9% win rate to break even
    ]
    
    if total_trades == 0:
        summary_data = {
            strategy_name: ['Total trades', 'Wins', 'Losses', 'Win Rate', 'Edge', 'Outcome']
        }
        for ratio, breakeven_rate in rrr_configs:
            summary_data[f'1:{ratio} RRR'] = [0, 0, 0, '0.0%', '0.0%', '0R']
        return pd.DataFrame(summary_data)
    
    
    summary_data = {
        strategy_name: ['Total trades', 'Wins', 'Losses', 'Win Rate', 'Edge', 'Outcome']
    }
    
    for ratio, breakeven_rate in rrr_configs:
        profitable = data_df[data_df['TP'] > ratio * data_df['SL']]
        wins = len(profitable)
        losses = total_trades - wins
        win_rate = (wins / total_trades) * 100
        edge = win_rate - breakeven_rate
        outcome = (wins * ratio) - losses
        
        summary_data[f'1:{ratio} RRR'] = [
            total_trades,
            wins,
            losses,
            f"{win_rate:.1f}%",
            f"{edge:.1f}%",
            f"{outcome}R"
        ]
    
    return pd.DataFrame(summary_data)

# Define strategy configurations
class Strategy:
    def __init__(self, name, filter_func, description=""):
        self.name = name
        self.filter_func = filter_func
        self.description = description
    
    def apply(self, df):
        """Apply the strategy filter to the dataframe."""
        return self.filter_func(df)

In [68]:
# Define all strategies in simple format
strategies = [
    Strategy(
        "Plain Strategy",
        lambda df: df,
        "All trades without any filtering"
    ),
    # Strategy(
    #     "1pip Pullback Strategy",
    #     lambda df: df[df['Pullback'] >= 1],
    #     "Trades with at least 1 pip pullback"
    # ),
]

In [69]:
# Generic strategy creator with custom lambdas
def create_custom_strategies(configs):
    """
    Generate multiple strategies with custom filter functions.
    
    Args:
        configs: List of tuples containing (name, filter_lambda, description)
                 where filter_lambda is a function that takes (df, value) as parameters
    
    Example:
        configs = [
            (1, lambda df, v: df[df['Pullback'] >= v], "at least {value} pip pullback"),
            (2, lambda df, v: df[df['Pullback'] >= v], "at least {value} pip pullback"),
        ]
    """
    custom_strategies = []
    for value, filter_func, description_template in configs:
        custom_strategies.append(
            Strategy(
                description_template,
                lambda df, v=value, f=filter_func: f(df, v),
                description_template.format(value=value) if '{value}' in description_template else description_template
            )
        )
    return custom_strategies

# Build tons of strategies
strategies.extend(create_custom_strategies([
    # Basic EMA and Direction Strategies
    (1, lambda df, v: df[df['EMA'] == df['Direction']], "EMA + Trade Direction"),
    (2, lambda df, v: df[df['EMA'] != df['Direction']], "EMA + Opposite Trade Direction"),
    (3, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS')], "EMA + BOS"),
    (4, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'CH')], "EMA + CH"),
    (5, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['BOS/CH'] == 'CH')], "Opposite EMA + CH"),
    (6, lambda df, v: df[(df['EMA'] != df['Direction']) & (df['BOS/CH'] == 'BOS')], "Opposite EMA + BOS"),
    (7, lambda df, v: df[df['BOS/CH'] == 'BOS'], "BOS"),
    (8, lambda df, v: df[df['BOS/CH'] == 'CH'], "CH"),

    # Risk Management Strategies (SL-based entry filters)
    (9, lambda df, v: df[df['SL'] >= 1], "SL >= 1 pip"),
    (10, lambda df, v: df[df['SL'] >= 2], "SL >= 2 pips"),
    (11, lambda df, v: df[df['SL'] >= 5], "SL >= 5 pips"),
    (12, lambda df, v: df[df['SL'] < 1], "SL < 1 pip"),
    (13, lambda df, v: df[(df['SL'] >= 1) & (df['SL'] <= 3)], "SL 1-3 pips"),
    (14, lambda df, v: df[(df['SL'] >= 3) & (df['SL'] <= 7)], "SL 3-7 pips"),
    (15, lambda df, v: df[(df['SL'] >= 7) & (df['SL'] <= 15)], "SL 7-15 pips"),
    (16, lambda df, v: df[df['SL'] >= 15], "SL >= 15 pips"),
    (17, lambda df, v: df[df['SL'] <= 0.5], "SL <= 0.5 pips"),
    (18, lambda df, v: df[df['SL'] >= 10], "SL >= 10 pips"),

    # Advanced Risk Management Strategies
    (19, lambda df, v: df[df['SL'] <= 2], "Conservative: SL <= 2 pips"),
    (20, lambda df, v: df[df['SL'] >= 3], "Moderate Risk: SL >= 3 pips"),
    (21, lambda df, v: df[df['SL'] >= 5], "Aggressive: SL >= 5 pips"),
    (22, lambda df, v: df[(df['SL'] >= 1) & (df['SL'] <= 3)], "Low Risk Zone: SL 1-3 pips"),
    (23, lambda df, v: df[(df['SL'] >= 3) & (df['SL'] <= 6)], "Medium Risk Zone: SL 3-6 pips"),

    # BOS/CH + SL Strategies (Risk-adjusted market structure)
    (24, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['SL'] <= 2)], "BOS + Conservative SL"),
    (25, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['SL'] >= 3)], "BOS + Moderate SL"),
    (26, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['SL'] <= 2)], "CH + Conservative SL"),
    (27, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['SL'] >= 3)], "CH + Moderate SL"),

    # Weekday Strategies
    (28, lambda df, v: df[df['Weekday'] == 'Monday'], "Monday Only"),
    (29, lambda df, v: df[df['Weekday'] == 'Tuesday'], "Tuesday Only"),
    (30, lambda df, v: df[df['Weekday'] == 'Wednesday'], "Wednesday Only"),
    (31, lambda df, v: df[df['Weekday'] == 'Thursday'], "Thursday Only"),
    (32, lambda df, v: df[df['Weekday'] == 'Friday'], "Friday Only"),
    
    # 30M Leg Trend Strategies
    (33, lambda df, v: df[df['30M\nLeg'].isin(['Above H', 'Above L'])], "Bullish 30M Trend"),
    (34, lambda df, v: df[df['30M\nLeg'].isin(['Below H', 'Below L'])], "Bearish 30M Trend"),
    (35, lambda df, v: df[df['30M\nLeg'] == 'Above H'], "30M Above High"),
    (36, lambda df, v: df[df['30M\nLeg'] == 'Above L'], "30M Above Low"),
    (37, lambda df, v: df[df['30M\nLeg'] == 'Below H'], "30M Below High"),
    (38, lambda df, v: df[df['30M\nLeg'] == 'Below L'], "30M Below Low"),
    
    # News Event Strategies
    (39, lambda df, v: df[df['News\nEvent'].isna()], "No News Events"),
    (40, lambda df, v: df[~df['News\nEvent'].isna()], "News Event Present"),
    (41, lambda df, v: df[(~df['News\nEvent'].isna()) & (df['HH Until\nNews'] >= 2)], "News in 2+ Hours"),
    (42, lambda df, v: df[(~df['News\nEvent'].isna()) & (df['HH Until\nNews'] <= 1)], "News in 1 Hour or Less"),
    
    # Combined Strategies: EMA + New Columns
    (43, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['30M\nLeg'].isin(['Above H', 'Above L']))], "EMA Match + Bullish 30M"),
    (44, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['30M\nLeg'].isin(['Below H', 'Below L']))], "EMA Match + Bearish 30M"),
    (45, lambda df, v: df[(df['EMA'] == df['Direction']) & (df['News\nEvent'].isna())], "EMA Match + No News"),
    
    # Combined Strategies: BOS/CH + New Columns
    (46, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['30M\nLeg'].isin(['Above H', 'Above L']))], "BOS + Bullish 30M"),
    (47, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['30M\nLeg'].isin(['Below H', 'Below L']))], "BOS + Bearish 30M"),
    (48, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['30M\nLeg'].isin(['Above H', 'Above L']))], "CH + Bullish 30M"),
    (49, lambda df, v: df[(df['BOS/CH'] == 'CH') & (df['30M\nLeg'].isin(['Below H', 'Below L']))], "CH + Bearish 30M"),
    (50, lambda df, v: df[(df['BOS/CH'] == 'BOS') & (df['News\nEvent'].isna())], "BOS + No News"),
    (51, lambda df, v: df[(df['BOS/CH'] == 'CH') & (~df['News\nEvent'].isna())], "CH + News Present"),
    
    # Combined Strategies: 30M Leg + Direction
    (52, lambda df, v: df[(df['Direction'] == 'Buy') & (df['30M\nLeg'].isin(['Above H', 'Above L']))], "Buy + Bullish 30M"),
    (53, lambda df, v: df[(df['Direction'] == 'Sell') & (df['30M\nLeg'].isin(['Below H', 'Below L']))], "Sell + Bearish 30M"),
    (54, lambda df, v: df[(df['Direction'] == 'Buy') & (df['30M\nLeg'].isin(['Below H', 'Below L']))], "Buy + Bearish 30M (Counter)"),
    (55, lambda df, v: df[(df['Direction'] == 'Sell') & (df['30M\nLeg'].isin(['Above H', 'Above L']))], "Sell + Bullish 30M (Counter)"),
    
    # Combined Strategies: News + Risk
    (56, lambda df, v: df[(df['News\nEvent'].isna()) & (df['SL'] <= 3)], "No News + Low Risk"),
    (57, lambda df, v: df[(~df['News\nEvent'].isna()) & (df['SL'] >= 5)], "News Event + High Risk"),
    (58, lambda df, v: df[(df['HH Until\nNews'] >= 2) & (df['EMA'] == df['Direction'])], "Far from News + EMA Match"),
    
    # Advanced Trend Alignment
    (59, lambda df, v: df[(df['30M\nLeg'] == 'Above H') & (df['Direction'] == 'Buy')], "Strong Bullish Alignment"),
    (60, lambda df, v: df[(df['30M\nLeg'] == 'Below L') & (df['Direction'] == 'Sell')], "Strong Bearish Alignment"),
    (61, lambda df, v: df[(df['30M\nLeg'] == 'Above L') & (df['BOS/CH'] == 'CH')], "Weak Bullish + CH"),
    (62, lambda df, v: df[(df['30M\nLeg'] == 'Below H') & (df['BOS/CH'] == 'CH')], "Weak Bearish + CH"),
]))

# Calculate all strategy results
strategy_results = {}
for strategy in strategies:
    filtered_df = strategy.apply(df)
    summary_df = calculate_rrr_stats(filtered_df, strategy.name)
    strategy_results[strategy.name] = summary_df

# Function to get top strategies for a specific RRR
def get_top_strategies_for_rrr(strategy_results, rrr_column, top_n=5):
    """Get top N strategies for a specific RRR based on outcome."""
    strategy_performance = []
    
    for strategy_name, df in strategy_results.items():
        # Extract metrics
        total_trades = df[rrr_column].iloc[0]
        wins = df[rrr_column].iloc[1]
        losses = df[rrr_column].iloc[2]
        win_rate = df[rrr_column].iloc[3]
        edge = df[rrr_column].iloc[4]
        outcome_str = df[rrr_column].iloc[5]
        outcome = int(outcome_str.replace('R', ''))
        
        strategy_performance.append({
            'Strategy': strategy_name,
            'Trades': total_trades,
            'Wins': wins,
            'Losses': losses,
            'Win Rate': win_rate,
            'Edge': edge,
            'Outcome': outcome_str,
            'outcome_value': outcome  # For sorting
        })
    
    # Sort by outcome and get top N
    top_strategies = sorted(strategy_performance, key=lambda x: x['outcome_value'], reverse=True)[:top_n]
    
    # Remove the sorting key from display
    for strat in top_strategies:
        del strat['outcome_value']
    
    return pd.DataFrame(top_strategies)

# Display top 5 strategies for each RRR in separate tables
display(HTML("<h2>Best Performing Strategies</h2>"))
rrr_configs = [
    ('1:1 RRR', '1:1'),
    ('1:2 RRR', '1:2'),
    ('1:3 RRR', '1:3'),
    ('1:4 RRR', '1:4'),
    ('1:5 RRR', '1:5'),
    ('1:10 RRR', '1:10')
]
for rrr_column, rrr_label in rrr_configs:
    top_5_df = get_top_strategies_for_rrr(strategy_results, rrr_column, top_n=5)
    top_5_df = top_5_df.rename(columns={'Strategy': 'Best ' + rrr_label + ' RRR Strategies'})
    top_5_df = top_5_df.style.set_properties(
        subset=['Best ' + rrr_label + ' RRR Strategies'], 
        **{'width': '250px'}
    )
    display(top_5_df)
    print()  # Add spacing between tables

,Best 1:1 RRR Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS + Bearish 30M,9,7,2,77.8%,27.8%,5R
1,Bearish 30M Trend,14,9,5,64.3%,14.3%,4R
2,BOS + No News,8,6,2,75.0%,25.0%,4R
3,Friday Only,5,4,1,80.0%,30.0%,3R
4,30M Below High,5,4,1,80.0%,30.0%,3R


,Best 1:2 RRR Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,EMA + Opposite Trade Direction,8,4,4,50.0%,17.0%,4R
1,BOS + Conservative SL,2,2,0,100.0%,67.0%,4R
2,Friday Only,5,3,2,60.0%,27.0%,4R
3,Bearish 30M Trend,14,6,8,42.9%,9.9%,4R
4,30M Below High,5,3,2,60.0%,27.0%,4R


,Best 1:3 RRR Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,Bearish 30M Trend,14,6,8,42.9%,17.9%,10R
1,EMA + Opposite Trade Direction,8,4,4,50.0%,25.0%,8R
2,BOS + No News,8,4,4,50.0%,25.0%,8R
3,Friday Only,5,3,2,60.0%,35.0%,7R
4,30M Below High,5,3,2,60.0%,35.0%,7R


,Best 1:4 RRR Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,EMA + Opposite Trade Direction,8,3,5,37.5%,17.5%,7R
1,CH + News Present,3,2,1,66.7%,46.7%,7R
2,Opposite EMA + CH,4,2,2,50.0%,30.0%,6R
3,SL 7-15 pips,5,2,3,40.0%,20.0%,5R
4,Friday Only,5,2,3,40.0%,20.0%,5R


,Best 1:5 RRR Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,Friday Only,5,2,3,40.0%,23.0%,7R
1,Weak Bearish + CH,1,1,0,100.0%,83.0%,5R
2,EMA + Opposite Trade Direction,8,2,6,25.0%,8.0%,4R
3,BOS + Conservative SL,2,1,1,50.0%,33.0%,4R
4,BOS + No News,8,2,6,25.0%,8.0%,4R


,Best 1:10 RRR Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,SL < 1 pip,0,0,0,0.0%,0.0%,0R
1,SL >= 15 pips,0,0,0,0.0%,0.0%,0R
2,SL <= 0.5 pips,0,0,0,0.0%,0.0%,0R
3,Thursday Only,0,0,0,0.0%,0.0%,0R
4,Weak Bullish + CH,0,0,0,0.0%,0.0%,0R


In [70]:
# Display all strategy results
display(HTML(f"<h2>Strategy Details ({len(strategy_results)})</h2>"))

for strategy_name, summary_df in strategy_results.items():
    # Style the first column to have a fixed width
    styled_df = summary_df.style.set_properties(
        subset=[strategy_name], 
        **{'width': '250px'}
    )
    display(styled_df)
    print('')  # Add spacing between tables

,Plain Strategy,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,24,24,24,24,24,24
1,Wins,12,8,7,4,3,0
2,Losses,12,16,17,20,21,24
3,Win Rate,50.0%,33.3%,29.2%,16.7%,12.5%,0.0%
4,Edge,0.0%,0.3%,4.2%,-3.3%,-4.5%,-9.0%
5,Outcome,0R,0R,4R,-4R,-6R,-24R


,EMA + Trade Direction,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,16,16,16,16,16,16
1,Wins,8,4,3,1,1,0
2,Losses,8,12,13,15,15,16
3,Win Rate,50.0%,25.0%,18.8%,6.2%,6.2%,0.0%
4,Edge,0.0%,-8.0%,-6.2%,-13.8%,-10.8%,-9.0%
5,Outcome,0R,-4R,-4R,-11R,-10R,-16R


,EMA + Opposite Trade Direction,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,8,8,8,8,8,8
1,Wins,4,4,4,3,2,0
2,Losses,4,4,4,5,6,8
3,Win Rate,50.0%,50.0%,50.0%,37.5%,25.0%,0.0%
4,Edge,0.0%,17.0%,25.0%,17.5%,8.0%,-9.0%
5,Outcome,0R,4R,8R,7R,4R,-8R


,EMA + BOS,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,13,13,13,13,13,13
1,Wins,7,3,3,1,1,0
2,Losses,6,10,10,12,12,13
3,Win Rate,53.8%,23.1%,23.1%,7.7%,7.7%,0.0%
4,Edge,3.8%,-9.9%,-1.9%,-12.3%,-9.3%,-9.0%
5,Outcome,1R,-4R,-1R,-8R,-7R,-13R


,EMA + CH,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,3,3,3,3,3,3
1,Wins,1,1,0,0,0,0
2,Losses,2,2,3,3,3,3
3,Win Rate,33.3%,33.3%,0.0%,0.0%,0.0%,0.0%
4,Edge,-16.7%,0.3%,-25.0%,-20.0%,-17.0%,-9.0%
5,Outcome,-1R,0R,-3R,-3R,-3R,-3R


...

,Far from News + EMA Match,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,4,4,4,4,4,4
1,Wins,1,0,0,0,0,0
2,Losses,3,4,4,4,4,4
3,Win Rate,25.0%,0.0%,0.0%,0.0%,0.0%,0.0%
4,Edge,-25.0%,-33.0%,-25.0%,-20.0%,-17.0%,-9.0%
5,Outcome,-2R,-4R,-4R,-4R,-4R,-4R


,Strong Bullish Alignment,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,2,2,2,2,2,2
1,Wins,1,1,0,0,0,0
2,Losses,1,1,2,2,2,2
3,Win Rate,50.0%,50.0%,0.0%,0.0%,0.0%,0.0%
4,Edge,0.0%,17.0%,-25.0%,-20.0%,-17.0%,-9.0%
5,Outcome,0R,1R,-2R,-2R,-2R,-2R


,Strong Bearish Alignment,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,7,7,7,7,7,7
1,Wins,5,3,3,2,1,0
2,Losses,2,4,4,5,6,7
3,Win Rate,71.4%,42.9%,42.9%,28.6%,14.3%,0.0%
4,Edge,21.4%,9.9%,17.9%,8.6%,-2.7%,-9.0%
5,Outcome,3R,2R,5R,3R,-1R,-7R


,Weak Bullish + CH,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,0,0,0,0,0,0
1,Wins,0,0,0,0,0,0
2,Losses,0,0,0,0,0,0
3,Win Rate,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
4,Edge,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
5,Outcome,0R,0R,0R,0R,0R,0R


,Weak Bearish + CH,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,1,1,1,1,1,1
1,Wins,1,1,1,1,1,0
2,Losses,0,0,0,0,0,1
3,Win Rate,100.0%,100.0%,100.0%,100.0%,100.0%,0.0%
4,Edge,50.0%,67.0%,75.0%,80.0%,83.0%,-9.0%
5,Outcome,1R,2R,3R,4R,5R,-1R
